# 03 SVHN SVM baseline：真實數字影像上的傳統 ML

前兩份 notebook 使用簡單、乾淨的手寫數字資料。這份開始切到 **SVHN cropped**：同樣是 0-9 數字分類，但圖片來自真實街景門牌，有顏色、背景、裁切、光線與雜訊。

本 notebook 不以追求最高分為目標，而是將 `02_digits_svm_baseline.ipynb` 的流程搬到更真實的影像資料上：

```text
資料 -> train/test split -> flatten -> standardize -> SVM/ML model -> evaluate -> error analysis
```

此處**不做降維**。PCA/t-SNE 在前面只用來看資料，此處要呈現 raw pixels + 傳統 ML 在真實影像上的限制。


## 0. 匯入套件與設定

Colab 通常已內建需要的套件。若缺少套件，再手動執行：

```text
%pip install scipy scikit-learn tensorflow
```


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

import tensorflow as tf

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

TRAIN_SAMPLES = 12000
TEST_SAMPLES = 3000


## 1. 下載 SVHN cropped

SVHN 是 Street View House Numbers。Stanford 原始 `.mat` 檔中，label `10` 代表數字 `0`，後面會轉回 `0`。


In [ ]:
SVHN_BASE_URL = 'http://ufldl.stanford.edu/housenumbers/'

train_path = tf.keras.utils.get_file(
    'svhn_train_32x32.mat',
    origin=SVHN_BASE_URL + 'train_32x32.mat',
    cache_dir='.',
    cache_subdir='data/svhn'
)

test_path = tf.keras.utils.get_file(
    'svhn_test_32x32.mat',
    origin=SVHN_BASE_URL + 'test_32x32.mat',
    cache_dir='.',
    cache_subdir='data/svhn'
)

print('train_path:', train_path)
print('test_path:', test_path)


## 2. 讀取資料與固定抽樣

為了執行時間穩定，此處固定抽取 train/test 子集。每個模型都用同一份資料，才算公平比較。


In [ ]:
def load_svhn_mat(path):
    data = loadmat(path)
    X = data['X']
    y = data['y'].reshape(-1)

    # 原始 shape: (height, width, channel, sample) -> (sample, height, width, channel)
    X = np.transpose(X, (3, 0, 1, 2))

    # SVHN 原始 label: 1-9 是原本數字，10 代表 0。
    y = y % 10
    return X, y.astype('int64')

def stratified_subset(X, y, n_samples, random_state=42):
    rng = np.random.default_rng(random_state)
    classes = np.unique(y)
    base = n_samples // len(classes)
    remainder = n_samples % len(classes)

    selected = []
    for rank, cls in enumerate(classes):
        cls_idx = np.where(y == cls)[0]
        take = base + (1 if rank < remainder else 0)
        take = min(take, len(cls_idx))
        selected.append(rng.choice(cls_idx, size=take, replace=False))

    selected = np.concatenate(selected)
    rng.shuffle(selected)
    return X[selected], y[selected]

X_train_full, y_train_full = load_svhn_mat(train_path)
X_test_full, y_test_full = load_svhn_mat(test_path)

X_train, y_train = stratified_subset(X_train_full, y_train_full, TRAIN_SAMPLES, RANDOM_STATE)
X_test, y_test = stratified_subset(X_test_full, y_test_full, TEST_SAMPLES, RANDOM_STATE + 1)

print('X_train:', X_train.shape, X_train.dtype)
print('X_test:', X_test.shape, X_test.dtype)
print('train class counts:', dict(zip(*np.unique(y_train, return_counts=True))))
print('test class counts:', dict(zip(*np.unique(y_test, return_counts=True))))


## 3. 觀察真實數字影像

可與前面的 `load_digits` 的 8x8 乾淨灰階手寫數字比較。SVHN 是 32x32 RGB 真實照片，分類難度自然高很多。


In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for ax, image, label in zip(axes.ravel(), X_train[:30], y_train[:30]):
    ax.imshow(image)
    ax.set_title(str(label))
    ax.axis('off')
plt.suptitle('SVHN examples: real-world street-view digits')
plt.tight_layout()
plt.show()

print('pixel min:', X_train.min())
print('pixel max:', X_train.max())
print('SVHN 是一般 8-bit RGB 影像，正規化到 0~1 時除以 255.0。')


## 4. Flatten 成 raw-pixel 特徵

傳統 ML 需要固定長度的 feature vector。本節直接把 `32x32x3` 攤平成 3072 維。

這是刻意設計的 baseline：不做 HOG、邊緣偵測、顏色統計等特徵工程，先觀察 raw pixels 的限制。


In [ ]:
X_train_float = X_train.astype('float32') / 255.0
X_test_float = X_test.astype('float32') / 255.0

X_train_flat = X_train_float.reshape(len(X_train_float), -1)
X_test_flat = X_test_float.reshape(len(X_test_float), -1)

print('image input:', X_train_float.shape)
print('flatten features:', X_train_flat.shape)


## 5. 傳統 ML baseline：RBF SVM

本節使用 `RBF SVM` 作為真實影像上的傳統 ML baseline，並在前處理中加入 `StandardScaler()`。

原因如下：

- RBF kernel 比線性模型更能處理複雜的非線性分類邊界。
- 模型仍只接收 flatten 後的 3072 維數值，無法直接利用像素的鄰近關係。
- 此結果可用來說明：即使改用較強的 SVM，面對真實影像資料仍會遭遇限制，且訓練成本也會上升。


In [ ]:
rbf_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(
        kernel='rbf',
        C=5.0,
        gamma='scale'
    ))
])

start = time.perf_counter()
rbf_svm.fit(X_train_flat, y_train)
elapsed = time.perf_counter() - start

train_pred = rbf_svm.predict(X_train_flat)
test_pred = rbf_svm.predict(X_test_flat)

results_df = pd.DataFrame([
    {
        'model': 'RBF SVM (raw pixels + StandardScaler)',
        'train_acc': accuracy_score(y_train, train_pred),
        'test_acc': accuracy_score(y_test, test_pred),
        'training_time_sec': elapsed,
    }
])
predictions = {
    'RBF SVM (raw pixels + StandardScaler)': test_pred
}

results_df


## 6. 看 SVM 錯在哪裡

本節透過錯誤案例說明資料複雜度對傳統 ML 的影響。

觀察重點：

- 背景是否形成干擾。
- 數字是否偏移或被裁切。
- 旁邊是否同時出現其他數字。
- 光線與顏色變化是否增加判別難度。


In [ ]:
best_model_name = results_df.iloc[0]['model']
best_pred = predictions[best_model_name]
print('best ML model:', best_model_name)
print(classification_report(y_test, best_pred))

fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, ax=ax, cmap='Blues', colorbar=False)
plt.title(f'{best_model_name} confusion matrix')
plt.show()

wrong_idx = np.where(best_pred != y_test)[0]
print('wrong predictions:', len(wrong_idx))

n_show = min(12, len(wrong_idx))
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, idx in zip(axes.ravel(), wrong_idx[:n_show]):
    ax.imshow(X_test[idx])
    ax.set_title(f't={y_test[idx]} p={best_pred[idx]}')
    ax.axis('off')
for ax in axes.ravel()[n_show:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 7. 小結：為什麼下一步進 DNN？

從前一份 `load_digits` 到本份 SVHN，可以整理出幾個重點：

1. **傳統機器學習不是不能用。** 在乾淨、小型、變化有限的資料上，SVM RBF 可以取得很好的分類效果。`load_digits` 就是典型例子：圖片解析度低、背景單純、灰階值範圍固定，因此模型比較容易找到有效的分類邊界。
2. **傳統 ML 很仰賴特徵工程。** 當資料變成真實照片後，模型直接吃 raw pixels 並不一定能理解「數字形狀」。實務上常需要先設計或萃取更好的特徵，例如邊緣、形狀、局部紋理、位置校正、背景去除或影像增強。
3. **資料量與影像複雜度會放大問題。** SVHN 雖然仍然只是 0-9 分類，但它包含彩色影像、背景干擾、光線差異、裁切偏移與不同字體。這些因素會讓 raw-pixel 傳統 ML 的效果明顯受限；當資料集更大、影像更複雜時，人工設計特徵的成本也會快速上升。
4. **神經網路提供另一種做法。** DNN 可以直接從資料中學習較複雜的非線性關係，不必完全依賴人工設計特徵。下一份 notebook 會使用同一份 SVHN 子集，把圖片正規化後交給 TensorFlow/Keras 的 Dense DNN，觀察模型是否能比傳統 ML 更準。

本節的目的不是否定 SVM，而是建立一個清楚的銜接：

> 當資料乾淨且特徵表達足夠時，傳統 ML 可以很有效；但當任務進入大量資料與複雜影像場景時，模型需要更強的表示學習能力。神經網路的價值，正在於讓模型從資料中自動學出更有用的表示，進而提升預測準確率。


## 課後練習與思考題

這一組練習用來觀察傳統機器學習方法在真實照片數字資料上的限制。SVHN 雖然仍然是 0-9 分類，但影像包含顏色、背景、裁切與光線變化，因此比 `load_digits` 更接近實務場景。

### 練習 1：調整資料量後重新訓練 RBF SVM

調整訓練資料量，觀察資料變多時 RBF SVM 的訓練時間與測試準確率如何改變。課堂練習可先使用較小子集，避免等待時間過長。

In [ ]:
# 練習 1 參考答案：調整資料量後重新訓練 RBF SVM

practice_train_samples = 3000
practice_test_samples = 1000

X_train_small, y_train_small = stratified_subset(
    X_train_full, y_train_full, practice_train_samples, RANDOM_STATE + 10
)
X_test_small, y_test_small = stratified_subset(
    X_test_full, y_test_full, practice_test_samples, RANDOM_STATE + 11
)

X_train_small_flat = (X_train_small.astype('float32') / 255.0).reshape(len(X_train_small), -1)
X_test_small_flat = (X_test_small.astype('float32') / 255.0).reshape(len(X_test_small), -1)

practice_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', C=5.0, gamma='scale'))
])

start = time.perf_counter()
practice_svm.fit(X_train_small_flat, y_train_small)
practice_time = time.perf_counter() - start

practice_train_acc = practice_svm.score(X_train_small_flat, y_train_small)
practice_test_acc = practice_svm.score(X_test_small_flat, y_test_small)

practice_result = pd.DataFrame([
    {
        'train_samples': practice_train_samples,
        'test_samples': practice_test_samples,
        'train_acc': practice_train_acc,
        'test_acc': practice_test_acc,
        'training_time_sec': practice_time,
    }
])
display(practice_result)



#### 練習 1 參考答案：調整資料量後重新訓練 RBF SVM

- 觀察說明：資料量增加通常有助於泛化，但 RBF SVM 的訓練時間也會明顯上升。


### 練習 2：調整 `C` 與 `gamma`

調整 RBF SVM 的 `C` 與 `gamma`，觀察模型複雜度對 train/test score 的影響。請同時看訓練時間，因為 SVHN 比 `load_digits` 大很多。

In [ ]:
# 練習 2 參考答案：調整 C 與 gamma，比較 train/test score

# RBF SVM 調參很耗時。課堂練習先使用較小子集，觀察趨勢即可。
tuning_train_samples = 3000
tuning_test_samples = 1000

X_tune_train, y_tune_train = stratified_subset(
    X_train_full, y_train_full, tuning_train_samples, RANDOM_STATE + 20
)
X_tune_test, y_tune_test = stratified_subset(
    X_test_full, y_test_full, tuning_test_samples, RANDOM_STATE + 21
)
X_tune_train_flat = (X_tune_train.astype('float32') / 255.0).reshape(len(X_tune_train), -1)
X_tune_test_flat = (X_tune_test.astype('float32') / 255.0).reshape(len(X_tune_test), -1)

param_candidates = [
    {'C': 1.0, 'gamma': 'scale'},
    {'C': 5.0, 'gamma': 'scale'},
    {'C': 10.0, 'gamma': 0.001},
]

rows = []
for params in param_candidates:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=params['C'], gamma=params['gamma']))
    ])
    start = time.perf_counter()
    model.fit(X_tune_train_flat, y_tune_train)
    elapsed = time.perf_counter() - start
    rows.append({
        'C': params['C'],
        'gamma': params['gamma'],
        'train_acc': model.score(X_tune_train_flat, y_tune_train),
        'test_acc': model.score(X_tune_test_flat, y_tune_test),
        'training_time_sec': elapsed,
    })

svm_tuning_df = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
display(svm_tuning_df)



#### 練習 2 參考答案：調整 C 與 gamma，比較 train/test score

- 觀察說明：C 或 gamma 變大時，模型可能更貼近訓練資料。
- 若 train_acc 上升但 test_acc 沒有同步上升，代表模型可能只是更會記住訓練資料，而不是更會辨識新圖片。


### 練習 3：比較 `load_digits` 與 SVHN 的資料條件

請整理兩個資料集的影像型態、資料形狀、像素範圍與主要難度，並說明為什麼它們的分數不能直接放在同一張排行榜比較。

#### 練習 3 參考答案：比較 load_digits 與 SVHN 的資料條件

| 資料集 | 影像型態 | 形狀 | 像素範圍 | 主要難度 |
| --- | --- | --- | --- | --- |
| `load_digits` | 乾淨灰階手寫數字 | `8 x 8 x 1` | `0-16` | 低解析度、背景乾淨、變化較小。 |
| `SVHN cropped` | 真實街景門牌照片 | `32 x 32 x 3` | `0-255`，本 notebook 正規化到 `0-1` | 彩色、背景干擾、裁切、光線與字體變化。 |

兩個資料集的影像難度、解析度、通道數與資料來源都不同，因此分數不能直接視為同一難度下的模型排名。本課程用 `load_digits` 建立 ML 流程直覺，再用 SVHN 展示真實影像會讓 raw-pixel 傳統 ML 遇到瓶頸。
